# SPX VIX-Style Term Structure — v0.7 Market-Close Production Notebook

**Methodology:** `v0.7_exchange_calendar_fred_sofr_market_close`

This notebook builds VIX-style implied-volatility term structures for custom target tenors using:

- **Options:** ThetaData SPX / SPXW option-chain quotes
- **Trading calendar:** exchange calendar / XNYS
- **Rates:** FRED SOFR
- **Quote timestamp:** official market close; if exact close is unavailable or malformed, fallback to 1, 2, 5, then 10 minutes before close
- **Expiration universe:** Friday-cycle, holiday-adjusted expirations
- **Target tenors:** 9, 12, 15, 18, 21, 24, 27, 30, 33 calendar days

Important production fixes included in this final notebook:

1. Early-close dates use actual market close, usually 13:00 ET.
2. Exact-close quote failures can fall back to nearby pre-close quotes.
3. Forward/K0 selection excludes junk zero/zero parity candidates.
4. Known vendor-unavailable chains can be excluded explicitly.


In [ ]:
# ============================================================
# Imports and global settings
# ============================================================

import io
import math
import requests
import numpy as np
import pandas as pd
import pandas_market_calendars as mcal

from pathlib import Path
from datetime import date, datetime, timedelta
from concurrent.futures import ThreadPoolExecutor, as_completed


# ThetaData v3 local terminal endpoint
BASE_URL_V3 = "http://127.0.0.1:25503/v3"


# Default reference time: 4:00 PM ET on normal trading days
# The production calculator overrides this per date using actual market close.
CALC_TIME_MS = 16 * 60 * 60 * 1000


# VIX methodology constants
MINUTES_PER_DAY = 1440
MINUTES_PER_YEAR = 525600
MINUTES_30_DAYS = 43200


# Rate settings
DEFAULT_RATE_SYMBOL = "SOFR"
DEFAULT_RISK_FREE_RATE = 0.05


# Target tenors for the first custom term-structure build
TARGET_TENOR_DAYS = [9, 12, 15, 18, 21, 24, 27, 30, 33]
PRIMARY_VALIDATION_TENOR_DAYS = 30


# Methodology labels
METHODOLOGY_VERSION = "v0.7_exchange_calendar_fred_sofr_market_close"
EXPIRATION_UNIVERSE = "friday_cycle_holiday_adjusted"
TRADING_CALENDAR_SOURCE = "exchange_calendar_XNYS"
RATE_SOURCE = "FRED_SOFR"
OPTION_SOURCE = "THETADATA_SPX_SPXW"


In [ ]:
# ============================================================
# Date and time helpers
# ============================================================

def yyyymmdd_to_date(x):
    """
    Convert 20240116 to datetime.date(2024, 1, 16).
    """
    s = str(int(x))
    return date(int(s[:4]), int(s[4:6]), int(s[6:8]))


def yyyymmdd_to_dash_string(x):
    """
    Convert 20240116 to '2024-01-16'.
    """
    return yyyymmdd_to_date(x).strftime("%Y-%m-%d")


def int_date_from_date(dt):
    """
    Convert datetime.date to YYYYMMDD integer.
    """
    return int(dt.strftime("%Y%m%d"))


def ms_to_time_string(ms_of_day):
    """
    Convert milliseconds after midnight to ThetaData v3 time string.

    Example:
        57600000 -> '16:00:00.000'
    """
    total_seconds = int(ms_of_day // 1000)

    hours = total_seconds // 3600
    minutes = (total_seconds % 3600) // 60
    seconds = total_seconds % 60

    return f"{hours:02d}:{minutes:02d}:{seconds:02d}.000"


def cache_time_label(calc_time_ms):
    """
    Convert calc time to compact label.

    Example:
        57600000 -> '160000'
    """
    time_str = ms_to_time_string(calc_time_ms)
    return time_str.replace(":", "").replace(".000", "")


def is_third_friday(dt):
    """
    True if date is the standard monthly SPX expiration Friday.
    """
    return dt.weekday() == 4 and 15 <= dt.day <= 21


QUOTE_TIME_LABEL = "market_close"


def ms_to_time_label(ms_of_day):
    """
    Convert milliseconds after midnight to HH:MM.
    """
    total_minutes = int(ms_of_day // (60 * 1000))
    hours = total_minutes // 60
    minutes = total_minutes % 60

    return f"{hours:02d}:{minutes:02d}"


In [ ]:
# ============================================================
# Project paths
# ============================================================

# This notebook usually runs from:
#     <project_root>/notebooks
#
# All data should be saved outside the notebooks folder:
#     <project_root>/data

CURRENT_DIR = Path.cwd()

if CURRENT_DIR.name.lower() == "notebooks":
    PROJECT_ROOT = CURRENT_DIR.parent
else:
    PROJECT_ROOT = CURRENT_DIR

DATA_DIR = PROJECT_ROOT / "data"
RAW_DATA_DIR = DATA_DIR / "raw"
PROCESSED_DATA_DIR = DATA_DIR / "processed"
EXTERNAL_DATA_DIR = DATA_DIR / "external"

CHAIN_CACHE_DIR = RAW_DATA_DIR / "thetadata_chains"
TERM_STRUCTURE_HISTORY_CSV = PROCESSED_DATA_DIR / "vix_term_structure_history.csv"
TERM_STRUCTURE_HISTORY_PARQUET = PROCESSED_DATA_DIR / "vix_term_structure_history.parquet"
SPX_TRADING_DATES_CSV = PROCESSED_DATA_DIR / "spx_trading_dates.csv"

RAW_DATA_DIR.mkdir(parents=True, exist_ok=True)
PROCESSED_DATA_DIR.mkdir(parents=True, exist_ok=True)
EXTERNAL_DATA_DIR.mkdir(parents=True, exist_ok=True)
CHAIN_CACHE_DIR.mkdir(parents=True, exist_ok=True)

print("Current directory:", CURRENT_DIR)
print("Project root:", PROJECT_ROOT)
print("Data directory:", DATA_DIR)
print("Processed history CSV:", TERM_STRUCTURE_HISTORY_CSV)
print("External data directory:", EXTERNAL_DATA_DIR)
print("SPX trading calendar CSV:", SPX_TRADING_DATES_CSV)
print("Raw chain cache:", CHAIN_CACHE_DIR)


## Data loaders

The v0.7 production setup uses ThetaData for SPX/SPXW options only. Trading dates come from an exchange calendar, and SOFR comes from FRED.


In [ ]:
# ============================================================
# ThetaData v3 expiration loader
# ============================================================

def list_expirations_v3(symbol, verbose=False):
    """
    Get option expirations from ThetaData v3.

    v3 endpoint returns CSV text like:
        symbol,expiration
        "SPX","2024-02-16"

    This function converts expiration dates to YYYYMMDD integers.
    """
    url = BASE_URL_V3 + "/option/list/expirations"
    params = {"symbol": symbol}

    r = requests.get(url, params=params, timeout=60)

    if verbose:
        print("URL:", r.url)
        print("Status code:", r.status_code)
        print(r.text[:1000])

    if r.status_code != 200:
        raise RuntimeError(
            f"ThetaData expiration request failed.\n"
            f"URL: {r.url}\n"
            f"Status code: {r.status_code}\n"
            f"Response text:\n{r.text[:2000]}"
        )

    df = pd.read_csv(io.StringIO(r.text))

    if df.empty:
        raise ValueError(f"No expirations returned for {symbol}")

    if "expiration" not in df.columns:
        raise ValueError(f"Could not find 'expiration' column. Columns are: {list(df.columns)}")

    expirations = (
        pd.to_datetime(df["expiration"])
        .dt.strftime("%Y%m%d")
        .astype(int)
        .tolist()
    )

    return sorted(expirations)


spx_exps = list_expirations_v3("SPX")
spxw_exps = list_expirations_v3("SPXW")

print("SPX expirations loaded:", len(spx_exps))
print("SPXW expirations loaded:", len(spxw_exps))
print("SPX sample:", spx_exps[:3], "...", spx_exps[-3:])
print("SPXW sample:", spxw_exps[:3], "...", spxw_exps[-3:])


In [ ]:
# ============================================================
# FRED SOFR loader
# ============================================================

FRED_SOFR_CSV = EXTERNAL_DATA_DIR / "fred_sofr_history.csv"
LEGACY_FRED_SOFR_CSV = PROCESSED_DATA_DIR / "fred_sofr_history.csv"
FRED_SOFR_URL = "https://fred.stlouisfed.org/graph/fredgraph.csv?id=SOFR"

_FRED_SOFR_CACHE = None


def load_fred_sofr_history(force_refresh=False, verbose=False):
    """
    Load SOFR history from FRED and cache it in memory.

    SOFR is quoted in percent form in the source file:
        5.32 means 5.32%

    This function converts it to decimal form:
        0.0532
    """
    global _FRED_SOFR_CACHE

    if _FRED_SOFR_CACHE is not None and not force_refresh:
        return _FRED_SOFR_CACHE.copy()

    if FRED_SOFR_CSV.exists() and not force_refresh:
        raw_df = pd.read_csv(FRED_SOFR_CSV)
    elif LEGACY_FRED_SOFR_CSV.exists() and not force_refresh:
        raw_df = pd.read_csv(LEGACY_FRED_SOFR_CSV)
        raw_df.to_csv(FRED_SOFR_CSV, index=False)
    else:
        raw_df = pd.read_csv(FRED_SOFR_URL)
        raw_df.to_csv(FRED_SOFR_CSV, index=False)

    sofr_df = raw_df.copy()
    sofr_df["observation_date"] = pd.to_datetime(sofr_df["observation_date"])
    sofr_df["trade_date"] = sofr_df["observation_date"].dt.strftime("%Y%m%d").astype(int)
    sofr_df["rate_pct"] = pd.to_numeric(sofr_df["SOFR"], errors="coerce")
    sofr_df = sofr_df.dropna(subset=["rate_pct"]).copy()
    sofr_df["rate_decimal"] = sofr_df["rate_pct"] / 100.0

    sofr_df = sofr_df[
        ["trade_date", "observation_date", "rate_pct", "rate_decimal"]
    ].sort_values("trade_date").reset_index(drop=True)

    _FRED_SOFR_CACHE = sofr_df.copy()

    if verbose:
        print("SOFR rows:", len(sofr_df))
        print("First SOFR date:", sofr_df["trade_date"].min())
        print("Last SOFR date:", sofr_df["trade_date"].max())
        print("Saved to:", FRED_SOFR_CSV)

    return sofr_df.copy()


def update_fred_sofr_history(force_refresh=False):
    """
    Refresh/load local FRED SOFR history and print a summary.
    """
    return load_fred_sofr_history(force_refresh=force_refresh, verbose=True)


def get_interest_rate_for_date_v3(symbol, trade_date, verbose=False):
    """
    Return SOFR for a trade date using local FRED history.

    Uses the latest available SOFR observation on or before trade_date.
    This avoids ThetaData interest-rate subscription limits before 2024.

    The function name is kept as get_interest_rate_for_date_v3 so the VIX engine
    can call the same interface used in earlier notebook versions.
    """
    symbol_upper = str(symbol).upper()

    if symbol_upper != "SOFR":
        raise ValueError(
            f"This v0.7 notebook currently supports only SOFR. Received: {symbol}"
        )

    sofr_df = load_fred_sofr_history(force_refresh=False, verbose=False)
    trade_date = int(trade_date)

    eligible = sofr_df[sofr_df["trade_date"] <= trade_date].copy()

    if eligible.empty:
        raise RuntimeError(
            f"No SOFR observation available on or before {trade_date}. "
            f"First available SOFR date is {sofr_df['trade_date'].min()}."
        )

    latest = eligible.iloc[-1]

    if verbose:
        print(
            f"Using SOFR {latest['rate_pct']:.4f}% "
            f"from {int(latest['trade_date'])} for trade date {trade_date}"
        )

    return float(latest["rate_decimal"])


In [ ]:
# ============================================================
# ThetaData v3 option-chain loader
# ============================================================

def get_chain_at_time(root, expi, trade_date, calc_time_ms, verbose=False):
    """
    Pull all strikes / both calls and puts for one SPX or SPXW expiration
    at a specific time using ThetaData v3.

    Returns dataframe with columns expected by the VIX math:
        root, expiration, strike, right, bid, ask, mid
    """
    request_time = ms_to_time_string(calc_time_ms)
    url = BASE_URL_V3 + "/option/history/quote"

    params = {
        "symbol": root,
        "expiration": yyyymmdd_to_dash_string(expi),
        "strike": "*",
        "right": "both",
        "start_date": yyyymmdd_to_dash_string(trade_date),
        "end_date": yyyymmdd_to_dash_string(trade_date),
        "start_time": request_time,
        "end_time": request_time,
        "interval": "1m",
        "format": "json",
    }

    r = requests.get(url, params=params, timeout=180)

    if verbose:
        print("URL:", r.url)
        print("Status code:", r.status_code)
        print(r.text[:1000])

    if r.status_code != 200:
        raise RuntimeError(
            f"ThetaData option-chain request failed.\n"
            f"URL: {r.url}\n"
            f"Status code: {r.status_code}\n"
            f"Response text:\n{r.text[:2000]}"
        )

    data = r.json()
    df = pd.DataFrame(data)

    if df.empty:
        raise ValueError(
            f"No data returned for {root} {expi} on {trade_date} at {request_time}"
        )

    df["root"] = df["symbol"]
    df["expiration"] = pd.to_datetime(df["expiration"]).dt.strftime("%Y%m%d").astype(int)

    right_map = {
        "CALL": "C",
        "PUT": "P",
        "C": "C",
        "P": "P",
    }

    df["right"] = df["right"].map(right_map)

    if df["right"].isna().any():
        bad_values = df["right"].unique()
        raise ValueError(f"Unknown option right values found: {bad_values}")

    df["bid"] = pd.to_numeric(df["bid"], errors="coerce")
    df["ask"] = pd.to_numeric(df["ask"], errors="coerce")
    df["strike"] = pd.to_numeric(df["strike"], errors="coerce")
    df["mid"] = (df["bid"] + df["ask"]) / 2

    keep_cols = [
        "root",
        "expiration",
        "strike",
        "right",
        "bid",
        "ask",
        "mid",
        "bid_size",
        "ask_size",
        "bid_exchange",
        "ask_exchange",
        "bid_condition",
        "ask_condition",
        "timestamp",
    ]

    return df[keep_cols].copy()


In [ ]:
# ============================================================
# SPX trading calendar from exchange calendar
# ============================================================

def yyyymmdd_int_to_iso(date_int):
    """
    Convert YYYYMMDD integer to YYYY-MM-DD string.
    """
    return pd.to_datetime(str(int(date_int)), format="%Y%m%d").strftime("%Y-%m-%d")


def get_spx_trading_dates_from_exchange_calendar(
    start_date,
    end_date,
    calendar_name="XNYS",
):
    """
    Get actual US equity trading sessions from an exchange calendar.

    This replaces ThetaData SPX index EOD as the trading-date source,
    because the user's ThetaData index EOD access starts later than the
    available options-history window.
    """
    cal = mcal.get_calendar(calendar_name)

    start_iso = yyyymmdd_int_to_iso(start_date)
    end_iso = yyyymmdd_int_to_iso(end_date)

    schedule = cal.schedule(
        start_date=start_iso,
        end_date=end_iso,
    )

    trading_dates = (
        pd.Series(schedule.index)
        .dt.strftime("%Y%m%d")
        .astype(int)
        .tolist()
    )

    return trading_dates


def update_spx_trading_dates_file(
    start_date,
    end_date,
    force_refresh=False,
    chunk_days=None,
    calendar_name="XNYS",
):
    """
    Build/update local SPX trading-date file using an exchange calendar.

    Use a calendar end date beyond your backfill end date so expiration selection
    can look ahead to future expirations and holiday Fridays.

    chunk_days is accepted for compatibility with earlier code but is not used.
    """
    if SPX_TRADING_DATES_CSV.exists() and not force_refresh:
        existing = pd.read_csv(SPX_TRADING_DATES_CSV)
    else:
        existing = pd.DataFrame(columns=["trade_date"])

    new_dates = pd.DataFrame({
        "trade_date": get_spx_trading_dates_from_exchange_calendar(
            start_date=start_date,
            end_date=end_date,
            calendar_name=calendar_name,
        )
    })

    combined = (
        pd.concat([existing[["trade_date"]], new_dates], ignore_index=True)
        .drop_duplicates()
        .sort_values("trade_date")
        .reset_index(drop=True)
    )

    combined.to_csv(SPX_TRADING_DATES_CSV, index=False)

    print(f"Saved SPX trading dates to: {SPX_TRADING_DATES_CSV}")
    print("Calendar source:", TRADING_CALENDAR_SOURCE)
    print("Rows:", len(combined))
    print("Min date:", combined["trade_date"].min())
    print("Max date:", combined["trade_date"].max())

    return combined


def load_spx_trading_dates():
    """
    Load locally saved SPX trading dates.
    """
    if not SPX_TRADING_DATES_CSV.exists():
        raise FileNotFoundError(
            f"{SPX_TRADING_DATES_CSV} does not exist yet. "
            "Run update_spx_trading_dates_file first."
        )

    df = pd.read_csv(SPX_TRADING_DATES_CSV)
    return sorted(df["trade_date"].astype(int).unique().tolist())


def get_spx_trade_dates_between(start_date, end_date):
    """
    Return actual SPX trading dates between start_date and end_date
    using the locally saved exchange trading calendar.
    """
    trading_dates = load_spx_trading_dates()

    start_date = int(start_date)
    end_date = int(end_date)

    return [
        int(d) for d in trading_dates
        if start_date <= int(d) <= end_date
    ]


## VIX-style calculation engine


In [ ]:
# ============================================================
# Core VIX-style variance math
# ============================================================

def settlement_minutes_after_midnight_et(root):
    """
    Approximate VIX methodology settlement timing.

    SPX monthly options are AM-settled.
    SPXW weekly options are PM-settled.
    """
    if root == "SPX":
        return 9 * 60 + 30       # 9:30 AM ET

    if root == "SPXW":
        return 16 * 60           # 4:00 PM ET

    raise ValueError(f"Unknown root: {root}")


def minutes_to_expiry_vix_method(trade_date, exp_yyyymmdd, root, calc_time_ms):
    """
    Minutes from calculation time to option settlement time.
    """
    trade_dt = yyyymmdd_to_date(trade_date)
    exp_dt = yyyymmdd_to_date(exp_yyyymmdd)

    calc_minutes_after_midnight = int(calc_time_ms // 60000)
    settlement_minutes = settlement_minutes_after_midnight_et(root)

    days_diff = (exp_dt - trade_dt).days

    return days_diff * MINUTES_PER_DAY + settlement_minutes - calc_minutes_after_midnight


def _prepare_call_put_tables(chain):
    """
    Create call and put tables indexed by strike.
    """
    df = chain.copy()

    df["strike"] = pd.to_numeric(df["strike"], errors="coerce")
    df["bid"] = pd.to_numeric(df["bid"], errors="coerce")
    df["ask"] = pd.to_numeric(df["ask"], errors="coerce")
    df["mid"] = pd.to_numeric(df["mid"], errors="coerce")

    df = df.dropna(subset=["strike", "bid", "ask", "mid", "right"])
    df = df[df["ask"] >= 0]
    df = df[df["bid"] >= 0]

    calls = (
        df[df["right"] == "C"]
        .sort_values("strike")
        .drop_duplicates(subset=["strike"], keep="last")
        .set_index("strike")
    )

    puts = (
        df[df["right"] == "P"]
        .sort_values("strike")
        .drop_duplicates(subset=["strike"], keep="last")
        .set_index("strike")
    )

    return calls, puts


def _select_otm_options_with_bid_rule(options_df, ascending=True):
    """
    Select OTM options using a simplified Cboe-style nonzero-bid rule.

    Walk away from ATM.
    Keep options with positive bid.
    Stop after two consecutive zero-bid options.
    """
    options_df = options_df.sort_values("strike", ascending=ascending)

    selected_rows = []
    consecutive_zero_bids = 0

    for _, row in options_df.iterrows():
        if row["bid"] <= 0:
            consecutive_zero_bids += 1

            if consecutive_zero_bids >= 2:
                break

            continue

        consecutive_zero_bids = 0
        selected_rows.append(row)

    if len(selected_rows) == 0:
        return pd.DataFrame(columns=options_df.columns)

    return pd.DataFrame(selected_rows)


def calc_single_term_variance(chain, minutes_to_expiry, r=DEFAULT_RISK_FREE_RATE):
    """
    Calculate VIX-style annualized variance for one expiration.
    """
    T = minutes_to_expiry / MINUTES_PER_YEAR

    calls, puts = _prepare_call_put_tables(chain)

    common_strikes = sorted(set(calls.index).intersection(set(puts.index)))

    if len(common_strikes) == 0:
        raise ValueError("No common call/put strikes found")

    parity_rows = []

    for K in common_strikes:
        call_mid = calls.loc[K, "mid"]
        put_mid = puts.loc[K, "mid"]
        diff = abs(call_mid - put_mid)

        parity_rows.append({
            "strike": K,
            "call_mid": call_mid,
            "put_mid": put_mid,
            "abs_call_put_diff": diff,
        })

    parity_df = pd.DataFrame(parity_rows)
    min_row = parity_df.loc[parity_df["abs_call_put_diff"].idxmin()]

    K_star = float(min_row["strike"])
    call_mid_star = float(min_row["call_mid"])
    put_mid_star = float(min_row["put_mid"])

    F = K_star + math.exp(r * T) * (call_mid_star - put_mid_star)

    all_strikes = sorted(set(calls.index).union(set(puts.index)))
    strikes_below_or_equal_forward = [K for K in all_strikes if K <= F]

    if len(strikes_below_or_equal_forward) == 0:
        raise ValueError("Could not find K0 below forward")

    K0 = max(strikes_below_or_equal_forward)

    put_rows = []

    for K in sorted([x for x in puts.index if x < K0], reverse=True):
        row = puts.loc[K].copy()
        row["strike"] = K
        row["QK"] = row["mid"]
        put_rows.append(row)

    put_otm_raw = pd.DataFrame(put_rows)
    put_otm = _select_otm_options_with_bid_rule(put_otm_raw, ascending=False)

    call_rows = []

    for K in sorted([x for x in calls.index if x > K0]):
        row = calls.loc[K].copy()
        row["strike"] = K
        row["QK"] = row["mid"]
        call_rows.append(row)

    call_otm_raw = pd.DataFrame(call_rows)
    call_otm = _select_otm_options_with_bid_rule(call_otm_raw, ascending=True)

    if K0 not in calls.index or K0 not in puts.index:
        raise ValueError(f"K0={K0} missing call or put quote")

    k0_row = calls.loc[K0].copy()
    k0_row["strike"] = K0
    k0_row["QK"] = (calls.loc[K0, "mid"] + puts.loc[K0, "mid"]) / 2
    k0_row["bid"] = (calls.loc[K0, "bid"] + puts.loc[K0, "bid"]) / 2
    k0_row["ask"] = (calls.loc[K0, "ask"] + puts.loc[K0, "ask"]) / 2
    k0_row["right"] = "K0_AVG"

    selected_options = pd.concat(
        [
            put_otm,
            pd.DataFrame([k0_row]),
            call_otm,
        ],
        ignore_index=True,
    )

    selected_options = selected_options.sort_values("strike").reset_index(drop=True)

    strikes = selected_options["strike"].astype(float).values
    QK = selected_options["QK"].astype(float).values

    delta_K = np.zeros(len(strikes))

    for i in range(len(strikes)):
        if i == 0:
            delta_K[i] = strikes[i + 1] - strikes[i]
        elif i == len(strikes) - 1:
            delta_K[i] = strikes[i] - strikes[i - 1]
        else:
            delta_K[i] = (strikes[i + 1] - strikes[i - 1]) / 2

    selected_options["delta_K"] = delta_K

    selected_options["contribution"] = (
        selected_options["delta_K"]
        / (selected_options["strike"] ** 2)
        * math.exp(r * T)
        * selected_options["QK"]
    )

    variance = (
        (2 / T) * selected_options["contribution"].sum()
        - (1 / T) * ((F / K0) - 1) ** 2
    )

    return {
        "variance": variance,
        "F": F,
        "K0": K0,
        "K_star": K_star,
        "T": T,
        "minutes": minutes_to_expiry,
        "num_options": len(selected_options),
        "selected_options": selected_options,
    }


In [ ]:
# ============================================================
# v0.7 expiration universe: Friday-cycle, holiday-adjusted
# ============================================================

def next_calendar_friday_after_date(dt):
    """
    Return the next calendar Friday on or after dt.

    Used to detect Friday-cycle expirations that are shifted earlier
    because the calendar Friday is a market holiday.
    """
    days_until_friday = (4 - dt.weekday()) % 7
    return dt + timedelta(days=days_until_friday)


def is_last_trading_day_before_closed_friday(exp_yyyymmdd, spx_trading_dates):
    """
    True if exp_yyyymmdd is the final SPX trading day before a closed calendar Friday.

    This handles Good Friday, July 4 on a Friday, Christmas on a Friday,
    and any other Friday market holiday.
    """
    exp_dt = yyyymmdd_to_date(exp_yyyymmdd)
    exp_int = int(exp_yyyymmdd)
    trading_date_set = set(int(x) for x in spx_trading_dates)

    if exp_int not in trading_date_set:
        return False

    if exp_dt.weekday() == 4:
        return False

    next_friday = next_calendar_friday_after_date(exp_dt)
    next_friday_int = int_date_from_date(next_friday)

    if next_friday_int in trading_date_set:
        return False

    check_dates = pd.date_range(
        start=exp_dt + timedelta(days=1),
        end=next_friday - timedelta(days=1),
        freq="D",
    )

    for d in check_dates:
        d_int = int(d.strftime("%Y%m%d"))
        if d_int in trading_date_set:
            return False

    return True


def is_friday_cycle_expiration_v7(exp_yyyymmdd, spx_trading_dates):
    """
    Friday-cycle expiration definition:

    1. Normal Friday expirations, but only if the expiration date is an SPX trading day.
    2. Non-Friday expirations that are the final SPX trading day
       before a closed calendar Friday.

    A closed Friday itself is not eligible.
    """
    exp_dt = yyyymmdd_to_date(exp_yyyymmdd)
    exp_int = int(exp_yyyymmdd)
    trading_date_set = set(int(x) for x in spx_trading_dates)

    if exp_int not in trading_date_set:
        return False

    if exp_dt.weekday() == 4:
        return True

    return is_last_trading_day_before_closed_friday(exp_yyyymmdd, spx_trading_dates)


def is_holiday_adjusted_monthly_expiration_v7(exp_yyyymmdd, spx_trading_dates):
    """
    True if this expiration is the holiday-adjusted version of a standard
    monthly third-Friday SPX expiration.
    """
    exp_dt = yyyymmdd_to_date(exp_yyyymmdd)

    if exp_dt.weekday() == 4:
        return False

    next_friday = next_calendar_friday_after_date(exp_dt)

    if not is_third_friday(next_friday):
        return False

    return is_last_trading_day_before_closed_friday(
        exp_yyyymmdd=exp_yyyymmdd,
        spx_trading_dates=spx_trading_dates,
    )


def preferred_root_for_expiration_v7(exp_yyyymmdd, spx_trading_dates):
    """
    Prefer SPX for standard monthly AM-settled expirations.

    Includes:
        - normal third-Friday SPX expirations
        - holiday-adjusted monthly SPX expirations shifted earlier

    Otherwise prefer SPXW for weekly/daily PM-settled options.
    """
    exp_dt = yyyymmdd_to_date(exp_yyyymmdd)

    if is_third_friday(exp_dt) and exp_yyyymmdd in spx_exps:
        return "SPX"

    if (
        is_holiday_adjusted_monthly_expiration_v7(exp_yyyymmdd, spx_trading_dates)
        and exp_yyyymmdd in spx_exps
    ):
        return "SPX"

    if exp_yyyymmdd in spxw_exps:
        return "SPXW"

    if exp_yyyymmdd in spx_exps:
        return "SPX"

    raise ValueError(f"Expiration {exp_yyyymmdd} not found in SPX or SPXW expiration lists")


def get_friday_cycle_expiration_candidates(
    trade_date,
    calc_time_ms=CALC_TIME_MS,
    min_minutes_to_expiry=1,
    spx_trading_dates=None,
):
    """
    Return future Friday-cycle SPX/SPXW expirations with days-to-expiry.

    Includes:
        - normal Friday expirations
        - holiday-adjusted expirations shifted earlier when Friday is closed
    """
    if spx_trading_dates is None:
        spx_trading_dates = load_spx_trading_dates()

    all_exps = sorted(set(spx_exps).union(set(spxw_exps)))
    rows = []

    for exp in all_exps:
        if not is_friday_cycle_expiration_v7(exp, spx_trading_dates):
            continue

        root = preferred_root_for_expiration_v7(exp, spx_trading_dates)

        minutes = minutes_to_expiry_vix_method(
            trade_date=trade_date,
            exp_yyyymmdd=exp,
            root=root,
            calc_time_ms=calc_time_ms,
        )

        if minutes <= min_minutes_to_expiry:
            continue

        rows.append({
            "root": root,
            "expiration": exp,
            "minutes": minutes,
            "days": minutes / MINUTES_PER_DAY,
        })

    candidates = pd.DataFrame(rows)

    if candidates.empty:
        raise ValueError(f"No future Friday-cycle expirations found for {trade_date}")

    return candidates.sort_values("minutes").reset_index(drop=True)


def choose_expiration_pair_for_target_days(
    trade_date,
    target_days,
    calc_time_ms=CALC_TIME_MS,
):
    """
    Choose the two Friday-cycle expirations that bracket a target tenor.
    """
    candidates = get_friday_cycle_expiration_candidates(
        trade_date=trade_date,
        calc_time_ms=calc_time_ms,
    )

    target_minutes = target_days * MINUTES_PER_DAY

    before = candidates[candidates["minutes"] <= target_minutes]
    after = candidates[candidates["minutes"] >= target_minutes]

    if before.empty:
        raise ValueError(f"No expiration before target {target_days}d for {trade_date}")

    if after.empty:
        raise ValueError(f"No expiration after target {target_days}d for {trade_date}")

    near_idx = before.index[-1]
    next_idx = after.index[0]

    if near_idx == next_idx:
        if next_idx + 1 < len(candidates):
            next_idx = next_idx + 1
        elif near_idx - 1 >= 0:
            near_idx = near_idx - 1
        else:
            raise ValueError(f"Could not form expiration pair for {target_days}d")

    near_exp = candidates.loc[near_idx].to_dict()
    next_exp = candidates.loc[next_idx].to_dict()

    return near_exp, next_exp


def get_required_chains_for_target_tenors(
    trade_date,
    target_tenor_days=TARGET_TENOR_DAYS,
    calc_time_ms=CALC_TIME_MS,
):
    """
    Return:
        required_by_tenor: one row per target tenor / near-next leg
        unique_chains: one row per unique option chain that must be pulled
    """
    rows = []

    for target_days in target_tenor_days:
        near_exp, next_exp = choose_expiration_pair_for_target_days(
            trade_date=trade_date,
            target_days=target_days,
            calc_time_ms=calc_time_ms,
        )

        rows.append({
            "target_days": target_days,
            "leg": "near",
            "root": near_exp["root"],
            "expiration": near_exp["expiration"],
            "minutes": near_exp["minutes"],
            "days": near_exp["days"],
        })

        rows.append({
            "target_days": target_days,
            "leg": "next",
            "root": next_exp["root"],
            "expiration": next_exp["expiration"],
            "minutes": next_exp["minutes"],
            "days": next_exp["days"],
        })

    required_by_tenor = pd.DataFrame(rows)

    unique_chains = (
        required_by_tenor[["root", "expiration", "minutes", "days"]]
        .drop_duplicates()
        .sort_values("minutes")
        .reset_index(drop=True)
    )

    return required_by_tenor, unique_chains


In [ ]:
# ============================================================
# Raw option-chain cache and parallel loading
# ============================================================

def get_chain_cache_path(root, trade_date, expi, calc_time_ms):
    """
    Local cache file path for one option chain.
    """
    time_label = cache_time_label(calc_time_ms)
    filename = f"{root}_{int(trade_date)}_{int(expi)}_{time_label}.pkl"

    return CHAIN_CACHE_DIR / filename


def get_chain_at_time_cached(
    root,
    expi,
    trade_date,
    calc_time_ms,
    force_refresh=False,
    verbose=False,
):
    """
    Cached wrapper around get_chain_at_time().

    If the chain exists locally, load it.
    Otherwise pull from ThetaData and save it.
    """
    cache_path = get_chain_cache_path(
        root=root,
        trade_date=trade_date,
        expi=expi,
        calc_time_ms=calc_time_ms,
    )

    if cache_path.exists() and not force_refresh:
        if verbose:
            print(f"Loading from cache: {cache_path}")

        return pd.read_pickle(cache_path)

    if verbose:
        print(f"Pulling from ThetaData: {root} {trade_date} {expi}")

    chain = get_chain_at_time(
        root=root,
        expi=expi,
        trade_date=trade_date,
        calc_time_ms=calc_time_ms,
        verbose=verbose,
    )

    chain.to_pickle(cache_path)

    if verbose:
        print(f"Saved to cache: {cache_path}")

    return chain


def pull_unique_chains_parallel_cached(
    trade_date,
    unique_chains,
    calc_time_ms=CALC_TIME_MS,
    max_workers=4,
    force_refresh=False,
    verbose=False,
):
    """
    Pull/load each unique option chain once, in parallel.

    Uses local cache:
        if cached file exists -> load it
        else -> pull from ThetaData and save it
    """
    chain_results = {}

    def _pull_one_chain(row):
        root = row["root"]
        expiration = int(row["expiration"])

        chain = get_chain_at_time_cached(
            root=root,
            expi=expiration,
            trade_date=trade_date,
            calc_time_ms=calc_time_ms,
            force_refresh=force_refresh,
            verbose=verbose,
        )

        return (root, expiration), chain

    rows = [row for _, row in unique_chains.iterrows()]

    with ThreadPoolExecutor(max_workers=max_workers) as executor:
        futures = [
            executor.submit(_pull_one_chain, row)
            for row in rows
        ]

        for future in as_completed(futures):
            key, chain = future.result()
            chain_results[key] = chain

    return chain_results


In [ ]:
# ============================================================
# Multi-tenor VIX-style term-structure engine
# ============================================================

def interpolate_variance_to_target_days(term_df, target_days):
    """
    Interpolate near and next term variances to a custom target tenor.

    Returns annualized variance in decimal form.
    Example:
        0.0190 means about 13.78 vol points, because sqrt(0.0190) * 100.
    """
    df = term_df.sort_values("minutes").reset_index(drop=True)

    if len(df) != 2:
        raise ValueError("term_df must have exactly two rows")

    N1 = df.loc[0, "minutes"]
    N2 = df.loc[1, "minutes"]

    T1 = N1 / MINUTES_PER_YEAR
    T2 = N2 / MINUTES_PER_YEAR

    var1 = df.loc[0, "variance"]
    var2 = df.loc[1, "variance"]

    target_minutes = target_days * MINUTES_PER_DAY

    if not (N1 <= target_minutes <= N2):
        raise ValueError(
            f"Target tenor {target_days} days is not bracketed by expirations: "
            f"near={N1 / MINUTES_PER_DAY:.2f} days, "
            f"next={N2 / MINUTES_PER_DAY:.2f} days"
        )

    interpolated_variance = (
        T1 * var1 * ((N2 - target_minutes) / (N2 - N1))
        + T2 * var2 * ((target_minutes - N1) / (N2 - N1))
    ) * (MINUTES_PER_YEAR / target_minutes)

    return interpolated_variance


def calculate_variance_for_unique_chains(unique_chains, chain_results, r):
    """
    Calculate one VIX-style variance for each unique option expiration.
    """
    rows = []
    calc_results = {}

    for _, row in unique_chains.iterrows():
        root = row["root"]
        expiration = int(row["expiration"])
        minutes = int(row["minutes"])
        days = float(row["days"])

        key = (root, expiration)
        chain = chain_results[key]

        calc = calc_single_term_variance(
            chain=chain,
            minutes_to_expiry=minutes,
            r=r,
        )

        calc_results[key] = calc

        rows.append({
            "root": root,
            "expiration": expiration,
            "minutes": minutes,
            "days": days,
            "variance": calc["variance"],
            "F": calc["F"],
            "K0": calc["K0"],
            "K_star": calc["K_star"],
            "num_options": calc["num_options"],
        })

    variance_table = pd.DataFrame(rows).sort_values("minutes").reset_index(drop=True)

    return variance_table, calc_results


def calculate_vix_term_structure_for_date_v7_cached(
    trade_date,
    target_tenor_days=TARGET_TENOR_DAYS,
    calc_time_ms=CALC_TIME_MS,
    rate_symbol=DEFAULT_RATE_SYMBOL,
    max_workers=4,
    force_refresh=False,
    return_details=True,
    verbose=False,
):
    """
    Calculate v0.7 VIX-style implied vol for multiple target tenors on one date.

    Uses:
        - same-date SOFR rate
        - Friday-cycle, holiday-adjusted SPX/SPXW expirations
        - parallel chain loading
        - local raw chain cache
    """
    rate = get_interest_rate_for_date_v3(
        symbol=rate_symbol,
        trade_date=trade_date,
    )

    required_by_tenor, unique_chains = get_required_chains_for_target_tenors(
        trade_date=trade_date,
        target_tenor_days=target_tenor_days,
        calc_time_ms=calc_time_ms,
    )

    chain_results = pull_unique_chains_parallel_cached(
        trade_date=trade_date,
        unique_chains=unique_chains,
        calc_time_ms=calc_time_ms,
        max_workers=max_workers,
        force_refresh=force_refresh,
        verbose=verbose,
    )

    variance_table, calc_results = calculate_variance_for_unique_chains(
        unique_chains=unique_chains,
        chain_results=chain_results,
        r=rate,
    )

    variance_lookup = {
        (row["root"], int(row["expiration"])): row
        for _, row in variance_table.iterrows()
    }

    output_rows = []

    for target_days in target_tenor_days:
        pair = required_by_tenor[
            required_by_tenor["target_days"] == target_days
        ].copy()

        if len(pair) != 2:
            raise ValueError(f"Expected two expiration rows for target {target_days}d")

        term_rows = []

        for _, leg_row in pair.iterrows():
            key = (leg_row["root"], int(leg_row["expiration"]))
            var_row = variance_lookup[key]

            term_rows.append({
                "term": leg_row["leg"],
                "root": leg_row["root"],
                "expiration": int(leg_row["expiration"]),
                "minutes": int(leg_row["minutes"]),
                "days": float(leg_row["days"]),
                "variance": float(var_row["variance"]),
                "F": float(var_row["F"]),
                "K0": float(var_row["K0"]),
                "num_options": int(var_row["num_options"]),
            })

        term_df = pd.DataFrame(term_rows).sort_values("minutes").reset_index(drop=True)

        implied_variance = interpolate_variance_to_target_days(
            term_df=term_df,
            target_days=target_days,
        )

        implied_vol = 100 * math.sqrt(implied_variance)

        output_rows.append({
            "trade_date": trade_date,
            "target_days": target_days,
            "rate_symbol": rate_symbol,
            "rate_pct": rate * 100,
            "implied_variance": implied_variance,
            "vix_style_vol": implied_vol,
            "near_root": term_df.loc[0, "root"],
            "near_expiration": term_df.loc[0, "expiration"],
            "near_days": term_df.loc[0, "days"],
            "near_variance": term_df.loc[0, "variance"],
            "next_root": term_df.loc[1, "root"],
            "next_expiration": term_df.loc[1, "expiration"],
            "next_days": term_df.loc[1, "days"],
            "next_variance": term_df.loc[1, "variance"],
        })

    result_df = pd.DataFrame(output_rows).sort_values("target_days").reset_index(drop=True)

    result = {
        "trade_date": trade_date,
        "rate_symbol": rate_symbol,
        "rate_decimal": rate,
        "rate_pct": rate * 100,
        "results_df": result_df,
    }

    if return_details:
        result["required_by_tenor"] = required_by_tenor
        result["unique_chains"] = unique_chains
        result["variance_table"] = variance_table
        result["chain_results"] = chain_results
        result["calc_results"] = calc_results

    return result


## Processed history and batch updates


In [ ]:
# ============================================================
# Processed history load / save / upsert
# ============================================================

def load_existing_term_structure_history():
    """
    Load existing processed history if it exists.

    Prefer parquet if available, otherwise CSV.
    """
    if TERM_STRUCTURE_HISTORY_PARQUET.exists():
        try:
            return pd.read_parquet(TERM_STRUCTURE_HISTORY_PARQUET)
        except Exception as e:
            print(f"Could not read parquet file. Falling back to CSV if available. Error: {e}")

    if TERM_STRUCTURE_HISTORY_CSV.exists():
        return pd.read_csv(TERM_STRUCTURE_HISTORY_CSV)

    return pd.DataFrame()


def save_term_structure_history(history_df):
    """
    Save processed history.

    Always writes CSV.
    Tries to write parquet if the environment supports it.
    """
    history_df = history_df.sort_values(["trade_date", "target_days"]).reset_index(drop=True)

    history_df.to_csv(TERM_STRUCTURE_HISTORY_CSV, index=False)

    try:
        history_df.to_parquet(TERM_STRUCTURE_HISTORY_PARQUET, index=False)
        print(f"Saved CSV: {TERM_STRUCTURE_HISTORY_CSV}")
        print(f"Saved parquet: {TERM_STRUCTURE_HISTORY_PARQUET}")
    except Exception as e:
        print(f"Saved CSV: {TERM_STRUCTURE_HISTORY_CSV}")
        print(f"Could not save parquet. CSV is still saved. Error: {e}")


def upsert_term_structure_history(new_rows_df):
    """
    Append/replace processed history rows.

    Replacement key:
        trade_date + target_days + methodology_version + quote_time
    """
    existing_df = load_existing_term_structure_history()

    if new_rows_df.empty:
        print("No new rows to save.")
        return existing_df

    key_cols = [
        "trade_date",
        "target_days",
        "methodology_version",
        "quote_time",
    ]

    missing_cols = [c for c in key_cols if c not in new_rows_df.columns]

    if missing_cols:
        raise ValueError(f"new_rows_df is missing key columns: {missing_cols}")

    if existing_df.empty:
        updated_df = new_rows_df.copy()
    else:
        existing_key = existing_df[key_cols].astype(str).agg("|".join, axis=1)
        new_key = new_rows_df[key_cols].astype(str).agg("|".join, axis=1)

        existing_without_replaced = existing_df[~existing_key.isin(set(new_key))].copy()

        updated_df = pd.concat(
            [existing_without_replaced, new_rows_df],
            ignore_index=True,
        )

    updated_df = updated_df.sort_values(["trade_date", "target_days"]).reset_index(drop=True)

    save_term_structure_history(updated_df)

    print("Rows in updated history:", len(updated_df))
    print("Min date:", updated_df["trade_date"].min())
    print("Max date:", updated_df["trade_date"].max())

    return updated_df


In [ ]:
# ============================================================
# Batch calculation and missing-date update
# ============================================================

def run_vix_term_structure_batch_v7(
    trade_dates,
    target_tenor_days=TARGET_TENOR_DAYS,
    calc_time_ms=CALC_TIME_MS,
    rate_symbol=DEFAULT_RATE_SYMBOL,
    max_workers=4,
    force_refresh=False,
    continue_on_error=True,
):
    """
    Calculate v0.7 multi-tenor VIX-style term structure for a list of trade dates.

    Returns:
        results_df: one row per trade_date / target_days
        errors_df: one row per failed trade_date, if any
    """
    all_results = []
    errors = []
    trade_dates = [int(d) for d in trade_dates]

    for i, trade_date in enumerate(trade_dates, start=1):
        print(f"[{i}/{len(trade_dates)}] Calculating {trade_date}...")

        try:
            result = calculate_vix_term_structure_for_date_v7_cached(
                trade_date=trade_date,
                target_tenor_days=target_tenor_days,
                calc_time_ms=calc_time_ms,
                rate_symbol=rate_symbol,
                max_workers=max_workers,
                force_refresh=force_refresh,
                return_details=False,
                verbose=False,
            )

            df = result["results_df"].copy()

            df["methodology_version"] = METHODOLOGY_VERSION
            df["expiration_universe"] = EXPIRATION_UNIVERSE
            df["trading_calendar_source"] = TRADING_CALENDAR_SOURCE
            df["rate_source"] = RATE_SOURCE
            df["option_source"] = OPTION_SOURCE
            df["quote_time"] = QUOTE_TIME_LABEL
            df["calc_time_ms"] = calc_time_ms

            all_results.append(df)

        except Exception as e:
            errors.append({
                "trade_date": trade_date,
                "error": str(e),
            })

            print(f"ERROR on {trade_date}: {e}")

            if not continue_on_error:
                raise

    if len(all_results) > 0:
        results_df = pd.concat(all_results, ignore_index=True)
        results_df = results_df.sort_values(["trade_date", "target_days"]).reset_index(drop=True)
    else:
        results_df = pd.DataFrame()

    errors_df = pd.DataFrame(errors)

    return results_df, errors_df


def get_completed_trade_dates_v7(history_df=None, target_tenor_days=TARGET_TENOR_DAYS):
    """
    Return trade dates that already have a complete set of target tenors
    for the v0.7 methodology version and quote time.
    """
    if history_df is None:
        history_df = load_existing_term_structure_history()

    if history_df.empty:
        return set()

    filtered = history_df[
        (history_df["methodology_version"] == METHODOLOGY_VERSION)
        & (history_df["quote_time"].astype(str) == str(QUOTE_TIME_LABEL))
    ].copy()

    if filtered.empty:
        return set()

    target_set = set(int(x) for x in target_tenor_days)
    completed_dates = set()

    for trade_date, group in filtered.groupby("trade_date"):
        tenor_set = set(int(x) for x in group["target_days"].unique())

        if target_set.issubset(tenor_set):
            completed_dates.add(int(trade_date))

    return completed_dates


def find_missing_trade_dates_v7(candidate_trade_dates, target_tenor_days=TARGET_TENOR_DAYS):
    """
    Given candidate trade dates, return only dates that do not yet have
    a complete v0.7 processed history.
    """
    history_df = load_existing_term_structure_history()

    completed_dates = get_completed_trade_dates_v7(
        history_df=history_df,
        target_tenor_days=target_tenor_days,
    )

    candidate_dates_clean = sorted(set(int(x) for x in candidate_trade_dates))

    return [
        d for d in candidate_dates_clean
        if d not in completed_dates
    ]


def update_term_structure_history_for_dates_v7(
    candidate_trade_dates,
    target_tenor_days=TARGET_TENOR_DAYS,
    calc_time_ms=CALC_TIME_MS,
    rate_symbol=DEFAULT_RATE_SYMBOL,
    max_workers=4,
    force_refresh=False,
    continue_on_error=True,
):
    """
    Update processed v0.7 term-structure history for missing dates only.
    """
    missing_dates = find_missing_trade_dates_v7(
        candidate_trade_dates=candidate_trade_dates,
        target_tenor_days=target_tenor_days,
    )

    print("Candidate dates:", len(set(candidate_trade_dates)))
    print("Missing/incomplete v0.7 dates:", len(missing_dates))

    if len(missing_dates) == 0:
        print("v0.7 history is already complete for the supplied dates.")
        return load_existing_term_structure_history(), pd.DataFrame()

    results_df, errors_df = run_vix_term_structure_batch_v7(
        trade_dates=missing_dates,
        target_tenor_days=target_tenor_days,
        calc_time_ms=calc_time_ms,
        rate_symbol=rate_symbol,
        max_workers=max_workers,
        force_refresh=force_refresh,
        continue_on_error=continue_on_error,
    )

    if not results_df.empty:
        updated_history_df = upsert_term_structure_history(results_df)
    else:
        updated_history_df = load_existing_term_structure_history()

    return updated_history_df, errors_df


def update_term_structure_history_for_range_v7(
    start_date,
    end_date,
    target_tenor_days=TARGET_TENOR_DAYS,
    calc_time_ms=CALC_TIME_MS,
    rate_symbol=DEFAULT_RATE_SYMBOL,
    max_workers=4,
    force_refresh=False,
    continue_on_error=True,
):
    """
    Update processed v0.7 history over an actual SPX trading-date range.
    """
    trade_dates = get_spx_trade_dates_between(
        start_date=start_date,
        end_date=end_date,
    )

    return update_term_structure_history_for_dates_v7(
        candidate_trade_dates=trade_dates,
        target_tenor_days=target_tenor_days,
        calc_time_ms=calc_time_ms,
        rate_symbol=rate_symbol,
        max_workers=max_workers,
        force_refresh=force_refresh,
        continue_on_error=continue_on_error,
    )


In [ ]:
# ============================================================
# Final v0.7 market-close production fixes
# ============================================================

# Final production labels
METHODOLOGY_VERSION = "v0.7_exchange_calendar_fred_sofr_market_close"
QUOTE_TIME_LABEL = "market_close"
TRADING_CALENDAR_SOURCE = "exchange_calendar_XNYS"
RATE_SOURCE = "FRED_SOFR"
OPTION_SOURCE = "THETADATA_SPX_SPXW"


def get_market_close_time_for_trade_date(trade_date, calendar_name="XNYS"):
    """
    Return the actual exchange market close for a trade date.

    Returns:
        close_time_label: e.g. "16:00" on normal days, "13:00" on early-close days
        close_time_ms: milliseconds after midnight ET
    """
    cal = mcal.get_calendar(calendar_name)
    date_iso = pd.to_datetime(str(int(trade_date)), format="%Y%m%d").strftime("%Y-%m-%d")

    schedule = cal.schedule(start_date=date_iso, end_date=date_iso)

    if schedule.empty:
        raise RuntimeError(f"No market calendar schedule found for {trade_date}")

    close_ts = schedule.iloc[0]["market_close"]
    close_ts_et = close_ts.tz_convert("America/New_York")

    close_time_label = close_ts_et.strftime("%H:%M")
    close_time_ms = (
        close_ts_et.hour * 60 * 60 * 1000
        + close_ts_et.minute * 60 * 1000
        + close_ts_et.second * 1000
    )

    return close_time_label, int(close_time_ms)


# --------------------------------------------------------------------------
# Known vendor-unavailable chains
# --------------------------------------------------------------------------
# These are not methodology exclusions. They are explicit vendor-data gaps found
# during production backfill. The calculator skips the unavailable chain and
# uses the next available Friday-cycle expiration to bracket the target tenor.
UNAVAILABLE_CHAINS = {
    # trade_date, root, expiration
    (20260223, "SPXW", 20260402),
}


try:
    _get_friday_cycle_expiration_candidates_unfiltered
except NameError:
    _get_friday_cycle_expiration_candidates_unfiltered = get_friday_cycle_expiration_candidates


def get_friday_cycle_expiration_candidates(
    trade_date,
    calc_time_ms=CALC_TIME_MS,
    min_minutes_to_expiry=1,
    spx_trading_dates=None,
):
    """
    Friday-cycle expiration candidates with known unavailable vendor chains removed.
    """
    candidates = _get_friday_cycle_expiration_candidates_unfiltered(
        trade_date=trade_date,
        calc_time_ms=calc_time_ms,
        min_minutes_to_expiry=min_minutes_to_expiry,
        spx_trading_dates=spx_trading_dates,
    ).copy()

    if candidates.empty:
        return candidates

    unavailable_mask = candidates.apply(
        lambda row: (
            int(trade_date),
            str(row["root"]),
            int(row["expiration"]),
        ) in UNAVAILABLE_CHAINS,
        axis=1,
    )

    removed = candidates[unavailable_mask].copy()

    if len(removed) > 0:
        print("Excluding unavailable ThetaData chains:")
        display(removed)

    candidates = candidates[~unavailable_mask].copy()

    if candidates.empty:
        raise ValueError(
            f"No Friday-cycle expirations remain after unavailable-chain filter for {trade_date}"
        )

    return candidates.sort_values("minutes").reset_index(drop=True)


# --------------------------------------------------------------------------
# Robust option selection and forward/K0 calculation
# --------------------------------------------------------------------------
def _select_otm_options_with_bid_rule(options_df, ascending=True):
    """
    Select OTM options using a simplified Cboe-style nonzero-bid rule.

    Handles empty input safely.
    """
    if options_df is None or len(options_df) == 0:
        return pd.DataFrame(columns=["strike", "bid", "ask", "mid", "QK"])

    if "strike" not in options_df.columns:
        return pd.DataFrame(columns=list(options_df.columns) + ["strike"])

    options_df = options_df.sort_values("strike", ascending=ascending)

    selected_rows = []
    consecutive_zero_bids = 0

    for _, row in options_df.iterrows():
        if row["bid"] <= 0:
            consecutive_zero_bids += 1

            if consecutive_zero_bids >= 2:
                break

            continue

        consecutive_zero_bids = 0
        selected_rows.append(row)

    if len(selected_rows) == 0:
        return pd.DataFrame(columns=options_df.columns)

    return pd.DataFrame(selected_rows)


def calc_single_term_variance(chain, minutes_to_expiry, r=DEFAULT_RISK_FREE_RATE):
    """
    Calculate VIX-style annualized variance for one expiration.

    Robust parity improvement:
        Exclude call/put parity candidates where either side has zero mid.
        This avoids selecting junk zero/zero deep-strike quotes as the forward.
    """
    T = minutes_to_expiry / MINUTES_PER_YEAR

    calls, puts = _prepare_call_put_tables(chain)
    common_strikes = sorted(set(calls.index).intersection(set(puts.index)))

    if len(common_strikes) == 0:
        raise ValueError("No common call/put strikes found")

    parity_rows = []

    for K in common_strikes:
        call_mid = float(calls.loc[K, "mid"])
        put_mid = float(puts.loc[K, "mid"])
        call_bid = float(calls.loc[K, "bid"])
        put_bid = float(puts.loc[K, "bid"])
        call_ask = float(calls.loc[K, "ask"])
        put_ask = float(puts.loc[K, "ask"])

        parity_rows.append({
            "strike": K,
            "call_mid": call_mid,
            "put_mid": put_mid,
            "call_bid": call_bid,
            "put_bid": put_bid,
            "call_ask": call_ask,
            "put_ask": put_ask,
            "abs_call_put_diff": abs(call_mid - put_mid),
        })

    parity_df = pd.DataFrame(parity_rows)

    parity_candidates = parity_df[
        (parity_df["call_mid"] > 0)
        & (parity_df["put_mid"] > 0)
        & (parity_df["call_ask"] >= parity_df["call_bid"])
        & (parity_df["put_ask"] >= parity_df["put_bid"])
    ].copy()

    if parity_candidates.empty:
        raise ValueError("No valid parity candidates after filtering zero/invalid quotes")

    min_row = parity_candidates.loc[parity_candidates["abs_call_put_diff"].idxmin()]

    K_star = float(min_row["strike"])
    call_mid_star = float(min_row["call_mid"])
    put_mid_star = float(min_row["put_mid"])

    F = K_star + math.exp(r * T) * (call_mid_star - put_mid_star)

    all_strikes = sorted(set(calls.index).union(set(puts.index)))
    strikes_below_or_equal_forward = [K for K in all_strikes if K <= F]

    if len(strikes_below_or_equal_forward) == 0:
        raise ValueError("Could not find K0 below forward")

    K0 = max(strikes_below_or_equal_forward)

    put_rows = []

    for K in sorted([x for x in puts.index if x < K0], reverse=True):
        row = puts.loc[K].copy()
        row["strike"] = K
        row["QK"] = row["mid"]
        put_rows.append(row)

    put_otm_raw = pd.DataFrame(put_rows)
    put_otm = _select_otm_options_with_bid_rule(put_otm_raw, ascending=False)

    call_rows = []

    for K in sorted([x for x in calls.index if x > K0]):
        row = calls.loc[K].copy()
        row["strike"] = K
        row["QK"] = row["mid"]
        call_rows.append(row)

    call_otm_raw = pd.DataFrame(call_rows)
    call_otm = _select_otm_options_with_bid_rule(call_otm_raw, ascending=True)

    if K0 not in calls.index or K0 not in puts.index:
        raise ValueError(f"K0={K0} missing call or put quote")

    k0_row = calls.loc[K0].copy()
    k0_row["strike"] = K0
    k0_row["QK"] = (calls.loc[K0, "mid"] + puts.loc[K0, "mid"]) / 2
    k0_row["bid"] = (calls.loc[K0, "bid"] + puts.loc[K0, "bid"]) / 2
    k0_row["ask"] = (calls.loc[K0, "ask"] + puts.loc[K0, "ask"]) / 2
    k0_row["right"] = "K0_AVG"

    selected_options = pd.concat(
        [
            put_otm,
            pd.DataFrame([k0_row]),
            call_otm,
        ],
        ignore_index=True,
    )

    selected_options = selected_options.sort_values("strike").reset_index(drop=True)

    if len(selected_options) < 2:
        raise ValueError("Insufficient selected options for variance calculation")

    strikes = selected_options["strike"].astype(float).values
    QK = selected_options["QK"].astype(float).values

    delta_K = np.zeros(len(strikes))

    for i in range(len(strikes)):
        if i == 0:
            delta_K[i] = strikes[i + 1] - strikes[i]
        elif i == len(strikes) - 1:
            delta_K[i] = strikes[i] - strikes[i - 1]
        else:
            delta_K[i] = (strikes[i + 1] - strikes[i - 1]) / 2

    selected_options["delta_K"] = delta_K

    selected_options["contribution"] = (
        selected_options["delta_K"]
        / (selected_options["strike"] ** 2)
        * math.exp(r * T)
        * selected_options["QK"]
    )

    variance = (
        (2 / T) * selected_options["contribution"].sum()
        - (1 / T) * ((F / K0) - 1) ** 2
    )

    return {
        "variance": variance,
        "F": F,
        "K0": K0,
        "K_star": K_star,
        "T": T,
        "minutes": minutes_to_expiry,
        "num_options": len(selected_options),
        "selected_options": selected_options,
    }


# --------------------------------------------------------------------------
# Market-close calculation wrapper with quote-time fallback
# --------------------------------------------------------------------------
try:
    _calculate_vix_term_structure_for_date_v7_cached_fixed_time
except NameError:
    _calculate_vix_term_structure_for_date_v7_cached_fixed_time = (
        calculate_vix_term_structure_for_date_v7_cached
    )


def calculate_vix_term_structure_for_date_v7_cached(
    trade_date,
    target_tenor_days=TARGET_TENOR_DAYS,
    calc_time_ms=CALC_TIME_MS,
    rate_symbol=DEFAULT_RATE_SYMBOL,
    max_workers=4,
    force_refresh=False,
    return_details=True,
    verbose=False,
):
    """
    Market-close wrapper with quote-time fallback.

    Normal target:
        actual market close, usually 16:00 or 13:00 on early-close days

    Fallback:
        if exact close fails, try 1, 2, 5, then 10 minutes before close.

    Audit columns:
        market_close_time = official exchange close time
        quote_time_used = actual option quote time used
        quote_time_offset_minutes = minutes before close
        calc_time_ms = actual milliseconds-after-midnight used
    """
    market_close_label, market_close_ms = get_market_close_time_for_trade_date(
        trade_date=trade_date,
    )

    fallback_minutes = [0, 1, 2, 5, 10]
    last_error = None

    for offset_minutes in fallback_minutes:
        actual_calc_time_ms = market_close_ms - offset_minutes * 60 * 1000
        quote_time_used = ms_to_time_label(actual_calc_time_ms)

        if verbose:
            print(
                f"Trying {trade_date} at {quote_time_used} "
                f"({offset_minutes} min before close)"
            )

        try:
            result = _calculate_vix_term_structure_for_date_v7_cached_fixed_time(
                trade_date=trade_date,
                target_tenor_days=target_tenor_days,
                calc_time_ms=actual_calc_time_ms,
                rate_symbol=rate_symbol,
                max_workers=max_workers,
                force_refresh=force_refresh,
                return_details=return_details,
                verbose=verbose,
            )

            if isinstance(result, dict) and "results_df" in result:
                result["results_df"] = result["results_df"].copy()
                result["results_df"]["market_close_time"] = market_close_label
                result["results_df"]["quote_time_used"] = quote_time_used
                result["results_df"]["quote_time_offset_minutes"] = offset_minutes
                result["results_df"]["calc_time_ms"] = actual_calc_time_ms

            elif isinstance(result, pd.DataFrame):
                result = result.copy()
                result["market_close_time"] = market_close_label
                result["quote_time_used"] = quote_time_used
                result["quote_time_offset_minutes"] = offset_minutes
                result["calc_time_ms"] = actual_calc_time_ms

            if verbose and offset_minutes > 0:
                print(f"Fallback succeeded for {trade_date} at {quote_time_used}")

            return result

        except Exception as e:
            last_error = e

            if verbose:
                print(f"Failed at {quote_time_used}: {e}")

    raise last_error


# --------------------------------------------------------------------------
# Batch runner that preserves actual quote-time audit columns
# --------------------------------------------------------------------------
def run_vix_term_structure_batch_v7(
    trade_dates,
    target_tenor_days=TARGET_TENOR_DAYS,
    calc_time_ms=CALC_TIME_MS,
    rate_symbol=DEFAULT_RATE_SYMBOL,
    max_workers=4,
    force_refresh=False,
    continue_on_error=True,
):
    """
    Calculate v0.7 multi-tenor VIX-style term structure for a list of trade dates.

    Uses market-close wrapper and preserves actual quote time used.
    """
    all_results = []
    errors = []
    trade_dates = [int(d) for d in trade_dates]

    for i, trade_date in enumerate(trade_dates, start=1):
        print(f"[{i}/{len(trade_dates)}] Calculating {trade_date}...")

        try:
            result = calculate_vix_term_structure_for_date_v7_cached(
                trade_date=trade_date,
                target_tenor_days=target_tenor_days,
                calc_time_ms=calc_time_ms,
                rate_symbol=rate_symbol,
                max_workers=max_workers,
                force_refresh=force_refresh,
                return_details=False,
                verbose=False,
            )

            df = result["results_df"].copy()

            df["methodology_version"] = METHODOLOGY_VERSION
            df["expiration_universe"] = EXPIRATION_UNIVERSE
            df["trading_calendar_source"] = TRADING_CALENDAR_SOURCE
            df["rate_source"] = RATE_SOURCE
            df["option_source"] = OPTION_SOURCE
            df["quote_time"] = QUOTE_TIME_LABEL

            # Preserve wrapper-provided audit fields if present.
            if "market_close_time" not in df.columns:
                market_close_label, market_close_ms = get_market_close_time_for_trade_date(trade_date)
                df["market_close_time"] = market_close_label
                df["quote_time_used"] = market_close_label
                df["quote_time_offset_minutes"] = 0
                df["calc_time_ms"] = market_close_ms

            all_results.append(df)

        except Exception as e:
            errors.append({
                "trade_date": trade_date,
                "error": str(e),
            })

            print(f"ERROR on {trade_date}: {e}")

            if not continue_on_error:
                raise

    if len(all_results) > 0:
        results_df = pd.concat(all_results, ignore_index=True)
        results_df = results_df.sort_values(["trade_date", "target_days"]).reset_index(drop=True)
    else:
        results_df = pd.DataFrame()

    errors_df = pd.DataFrame(errors)

    return results_df, errors_df


print("Final v0.7 production settings loaded:")
print("Methodology version:", METHODOLOGY_VERSION)
print("Quote time label:", QUOTE_TIME_LABEL)
print("Trading calendar source:", TRADING_CALENDAR_SOURCE)
print("Rate source:", RATE_SOURCE)
print("Option source:", OPTION_SOURCE)


In [ ]:
# ============================================================
# Validation and audit helpers
# ============================================================

def validate_history_range_v7(
    start_date,
    end_date,
    target_tenor_days=TARGET_TENOR_DAYS,
):
    """
    Validate processed history completeness for a date range.

    Returns:
        summary_df, bad_dates_df, duplicates_df
    """
    history_df = load_existing_term_structure_history()

    trade_dates = get_spx_trade_dates_between(
        start_date=start_date,
        end_date=end_date,
    )

    history_slice = history_df[
        (history_df["trade_date"] >= int(start_date))
        & (history_df["trade_date"] <= int(end_date))
        & (history_df["methodology_version"] == METHODOLOGY_VERSION)
        & (history_df["quote_time"].astype(str) == str(QUOTE_TIME_LABEL))
    ].copy()

    summary = {
        "start_date": int(start_date),
        "end_date": int(end_date),
        "expected_trade_dates": len(trade_dates),
        "actual_trade_dates": history_slice["trade_date"].nunique() if not history_slice.empty else 0,
        "expected_rows": len(trade_dates) * len(target_tenor_days),
        "actual_rows": len(history_slice),
    }

    summary_df = pd.DataFrame([summary])

    if history_slice.empty:
        bad_dates_df = pd.DataFrame({
            "trade_date": trade_dates,
            "rows": 0,
            "unique_tenors": 0,
            "min_tenor": np.nan,
            "max_tenor": np.nan,
        })
        duplicates_df = pd.DataFrame()
        return summary_df, bad_dates_df, duplicates_df

    counts = (
        history_slice
        .groupby("trade_date")
        .agg(
            rows=("target_days", "count"),
            unique_tenors=("target_days", "nunique"),
            min_tenor=("target_days", "min"),
            max_tenor=("target_days", "max"),
        )
        .reset_index()
    )

    expected_dates_df = pd.DataFrame({"trade_date": trade_dates})
    counts = expected_dates_df.merge(counts, on="trade_date", how="left").fillna({
        "rows": 0,
        "unique_tenors": 0,
    })

    bad_dates_df = counts[
        (counts["rows"] != len(target_tenor_days))
        | (counts["unique_tenors"] != len(target_tenor_days))
        | (counts["min_tenor"] != min(target_tenor_days))
        | (counts["max_tenor"] != max(target_tenor_days))
    ].copy()

    duplicate_check = (
        history_slice
        .groupby(["trade_date", "target_days", "methodology_version", "quote_time"])
        .size()
        .reset_index(name="count")
    )

    duplicates_df = duplicate_check[duplicate_check["count"] > 1].copy()

    return summary_df, bad_dates_df, duplicates_df


def audit_closed_friday_expirations(start_date, end_date, spx_trading_dates=None):
    """
    Find closed calendar Fridays and show the prior trading-day expiration handling.
    """
    if spx_trading_dates is None:
        spx_trading_dates = load_spx_trading_dates()

    trading_date_set = set(int(x) for x in spx_trading_dates)
    all_expirations = set(spx_exps).union(set(spxw_exps))

    start_ts = pd.to_datetime(str(int(start_date)), format="%Y%m%d")
    end_ts = pd.to_datetime(str(int(end_date)), format="%Y%m%d")

    all_fridays = pd.date_range(start=start_ts, end=end_ts, freq="W-FRI")

    rows = []

    for friday in all_fridays:
        friday_int = int(friday.strftime("%Y%m%d"))

        if friday_int in trading_date_set:
            continue

        prior_trading_dates = [d for d in trading_date_set if d < friday_int]

        if len(prior_trading_dates) == 0:
            continue

        adjusted_exp = max(prior_trading_dates)
        expiration_exists = adjusted_exp in all_expirations

        if expiration_exists:
            if adjusted_exp in spx_exps and adjusted_exp in spxw_exps:
                listed_roots = "SPX,SPXW"
            elif adjusted_exp in spx_exps:
                listed_roots = "SPX"
            else:
                listed_roots = "SPXW"
        else:
            listed_roots = None

        included_by_v7 = is_friday_cycle_expiration_v7(
            adjusted_exp,
            spx_trading_dates,
        )

        preferred_root = None

        if expiration_exists and included_by_v7:
            preferred_root = preferred_root_for_expiration_v7(
                adjusted_exp,
                spx_trading_dates,
            )

        rows.append({
            "closed_friday": friday_int,
            "adjusted_expiration": adjusted_exp,
            "adjusted_expiration_weekday": yyyymmdd_to_date(adjusted_exp).strftime("%A"),
            "expiration_exists": expiration_exists,
            "listed_roots": listed_roots,
            "included_by_v7": included_by_v7,
            "preferred_root": preferred_root,
        })

    return pd.DataFrame(rows)


## Usage cells

Run the definition cells above first. Then run these usage cells manually.

Recommended production sequence:

1. Build/refresh the trading calendar with a future buffer.
2. Load/refresh FRED SOFR.
3. Run the smoke test.
4. Run one backfill chunk at a time.
5. Validate that chunk before moving on.


In [ ]:
# ============================================================
# Build or refresh full backfill trading calendar
# ============================================================

FULL_BACKFILL_START_DATE = 20180625
FULL_BACKFILL_END_DATE = 20260625
CALENDAR_END_DATE_WITH_BUFFER = 20261231

spx_trading_dates_full_df = update_spx_trading_dates_file(
    start_date=FULL_BACKFILL_START_DATE,
    end_date=CALENDAR_END_DATE_WITH_BUFFER,
    force_refresh=True,
)

spx_trading_dates_full = load_spx_trading_dates()

print("Trading dates loaded:", len(spx_trading_dates_full))
print("First:", min(spx_trading_dates_full))
print("Last:", max(spx_trading_dates_full))
display(spx_trading_dates_full_df.tail())


In [ ]:
# ============================================================
# Load or refresh FRED SOFR history
# ============================================================

sofr_df = update_fred_sofr_history(force_refresh=False)
display(sofr_df.tail())


In [ ]:
# ============================================================
# Confirm active processed history status
# ============================================================

history_df = load_existing_term_structure_history()
print("Rows loaded:", len(history_df))

if not history_df.empty:
    display(history_df.tail())


In [ ]:
# ============================================================
# v0.7 smoke test: early and recent dates
# ============================================================

SMOKE_TEST_DATES = [20180625, FULL_BACKFILL_END_DATE]

for smoke_test_date in SMOKE_TEST_DATES:
    smoke_test_result = calculate_vix_term_structure_for_date_v7_cached(
        trade_date=smoke_test_date,
        target_tenor_days=TARGET_TENOR_DAYS,
        calc_time_ms=CALC_TIME_MS,
        rate_symbol=DEFAULT_RATE_SYMBOL,
        max_workers=4,
        force_refresh=False,
        return_details=True,
        verbose=False,
    )

    print("Smoke test date:", smoke_test_date)
    display(smoke_test_result["results_df"])


In [ ]:
# ============================================================
# Production backfill: one chunk at a time
# ============================================================

# Start with 2018 partial year. After validation passes, change these dates
# to 2019, 2020, 2021, etc.
BACKFILL_START_DATE = 20180625
BACKFILL_END_DATE = 20181231

history_df, errors_df = update_term_structure_history_for_range_v7(
    start_date=BACKFILL_START_DATE,
    end_date=BACKFILL_END_DATE,
    target_tenor_days=TARGET_TENOR_DAYS,
    calc_time_ms=CALC_TIME_MS,
    rate_symbol=DEFAULT_RATE_SYMBOL,
    max_workers=4,
    force_refresh=False,
    continue_on_error=True,
)

print("Rows in history:", len(history_df))
print("Errors:", len(errors_df))
display(errors_df)


In [ ]:
# ============================================================
# Validate processed history for the current backfill range
# ============================================================

summary_df, bad_dates_df, duplicates_df = validate_history_range_v7(
    start_date=BACKFILL_START_DATE,
    end_date=BACKFILL_END_DATE,
    target_tenor_days=TARGET_TENOR_DAYS,
)

display(summary_df)

print("Bad/incomplete dates:", len(bad_dates_df))
display(bad_dates_df)

print("Duplicate key rows:", len(duplicates_df))
display(duplicates_df)


## Suggested full backfill chunks

The v0.7 market-close backfill can be run in 1-year or 2-year chunks. The completed production backfill used these chunks:

- `20180625` to `20181231`
- `20190101` to `20201231`
- `20210101` to `20221231`
- `20230101` to `20241231`
- `20250101` to `20260625`

Validate after each chunk before moving to the next one.


## Final full-history validation

After the full backfill finishes, use the existing `validate_history_range_v7` helper over the full range.


In [ ]:
# ============================================================
# Validate full v0.7 market-close history
# ============================================================

summary_df, bad_dates_df, duplicates_df = validate_history_range_v7(
    start_date=FULL_BACKFILL_START_DATE,
    end_date=FULL_BACKFILL_END_DATE,
    target_tenor_days=TARGET_TENOR_DAYS,
)

display(summary_df)

print("Bad/incomplete full-history dates:", len(bad_dates_df))
display(bad_dates_df)

print("Duplicate key rows:", len(duplicates_df))
display(duplicates_df)

history_df = load_existing_term_structure_history()
full_slice = history_df[
    (history_df["trade_date"] >= FULL_BACKFILL_START_DATE)
    & (history_df["trade_date"] <= FULL_BACKFILL_END_DATE)
    & (history_df["methodology_version"] == METHODOLOGY_VERSION)
].copy()

print("Quote-time offsets full history:")
display(full_slice["quote_time_offset_minutes"].value_counts().sort_index())

print("Market-close times full history:")
display(full_slice["market_close_time"].value_counts())

print("Methodology versions:")
display(full_slice["methodology_version"].value_counts())

print("Rate sources:")
display(full_slice["rate_source"].value_counts())

print("Option sources:")
display(full_slice["option_source"].value_counts())
